In [3]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

# -------- Settings --------
data_path = "./"
csv_files = [
    "aya-expanse-32b_test.csv",
    "gemma3-12b_test.csv",
    "gemma3-27b_test.csv",
    "gpt4o_test.csv",
    "llama3.1_test.csv",
    "llama3.2-latest_test.csv",
    "llama3.3-latest_test.csv",
    "mistral-small-3.2-24b_test.csv",
    "mistral-small3.1-latest_test.csv",
    "qwen2.5-14b_test.csv",
    "qwen2.5-32b_test.csv",
    "qwen2.5-72b_test.csv",
]

# -------- Helpers --------
def extract_choice(ans: str) -> str:
    if not isinstance(ans, str):
        return None
    m = re.search(r"\b([ABC])\b", ans)
    return m.group(1) if m else None

# Mapping A/B/C → Normative positioning
norm_map = {"A": "Conformity", "B": "Assertion", "C": "Compromise"}

# -------- Collect --------
rows = []
for file in csv_files:
    path = os.path.join(data_path, file)
    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)
    model = file.replace("_test.csv", "")
    if "answer" not in df.columns:
        continue

    tmp = df[["scenario_code", "answer"]].copy()
    tmp["model"] = model
    tmp["choice"] = tmp["answer"].apply(extract_choice)
    tmp["normative"] = tmp["choice"].map(norm_map)
    
    # Optional: role from scenario_code (adjust if needed)
    tmp["role"] = tmp["scenario_code"].apply(
        lambda x: "student/advisor" if "EDU" in x else 
                  "patient/doctor" if "MED" in x else 
                  "other"
    )

    rows.append(tmp)

all_df = pd.concat(rows, ignore_index=True)

# -------- Normative distribution --------
counts = all_df.groupby(["model","normative"]).size().unstack(fill_value=0)
props = counts.div(counts.sum(axis=1), axis=0).round(3)

# Save
counts.to_csv("normative_counts.csv")
props.to_csv("normative_proportions.csv")

print("\n=== Normative Positioning Counts ===")
print(counts)
print("\n=== Normative Positioning Proportions ===")
print(props)

# -------- Plot proportions --------
# Desired order: Conformity, Assertion, Compromise
plot_order = ["Conformity", "Assertion", "Compromise"]
props_plot = props.reindex(columns=plot_order)

# Matching pastel colors for that order:
# Conformity (light blue), Assertion (light orange), Compromise (light green)
pastel_colors = ["#aec7e8", "#ffbb78", "#98df8a"]

props_plot.plot(kind="bar", stacked=True, figsize=(12,6), color=pastel_colors)
plt.title("Normative Positioning Distribution (Conformity, Assertion, Compromise)")
plt.ylabel("Proportion")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Normative stance")
plt.tight_layout()
plt.savefig("normative_positioning_distribution.pdf")  # ✅ save as PDF
plt.close()



=== Normative Positioning Counts ===
normative                Assertion  Compromise  Conformity
model                                                     
aya-expanse-32b                 28          70          21
gemma3-12b                      25          84          11
gemma3-27b                      22          84          14
gpt4o                           24          83          13
llama3.1                        45          60          15
llama3.2-latest                 47          63          10
llama3.3-latest                 42          64          14
mistral-small-3.2-24b           36          68          16
mistral-small3.1-latest         30          76          14
qwen2.5-14b                     21          92           7
qwen2.5-32b                     30          76          14
qwen2.5-72b                     30          80          10

=== Normative Positioning Proportions ===
normative                Assertion  Compromise  Conformity
model                             